# 09 — One training step

Three losses, added:

- cross-entropy on topic
- cross-entropy on urgency level
- binary cross-entropy on the reply logit

There is no next-token shift. The label is a decision about the whole window.

**You implement:** `train_step(model, batch, optimizer) -> float`.

`batch` has `input_ids`, `mask`, `topic` (long), `urgency` (long), `reply` (float 0/1). The model returns an object with `.topic`, `.urgency`, and `.reply` logits.



In [ ]:
from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import Tensor


def train_step(model, batch: dict[str, Tensor], optimizer: torch.optim.Optimizer) -> float:
    raise NotImplementedError("train_step")



<details>
<summary><b>Solution</b> (try yourself first — click to expand)</summary>

```python
import torch
import torch.nn.functional as F
from torch import Tensor

def train_step(model, batch: dict[str, Tensor], optimizer: torch.optim.Optimizer) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    out = model(batch["input_ids"], batch["mask"])
    loss = (
        F.cross_entropy(out.topic, batch["topic"])
        + F.cross_entropy(out.urgency, batch["urgency"])
        + F.binary_cross_entropy_with_logits(out.reply, batch["reply"])
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return float(loss.detach())
```

Copy into the stub cell above, then run **Check**.

</details>



## Check



In [ ]:
import torch
from buka_rs.config import OneConfig
from buka_rs.data import Example, collate
from buka_rs.model import OneModel

cfg = OneConfig(name="micro", d_model=32, n_layers=1, n_heads=4, d_ff=64, max_seq_len=32)
model = OneModel(cfg)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
rows = [Example("server is down tonight?", "ops", 2, True)] * 4
batch = collate(rows, cfg.topics, 24, cfg.pad_id, torch.device("cpu"))
before = [p.detach().clone() for p in model.parameters()]
loss = train_step(model, batch, opt)
assert loss > 0
assert any(not torch.equal(a, b) for a, b in zip(before, model.parameters()))
print("step loss", round(loss, 4))

